<h2>Download Initial Dependencies</h2>

In [1]:
import numpy as np
import pandas as pd
import json
import math
from tqdm import tqdm
from qiskit.quantum_info import SparsePauliOp, Operator, Pauli, Clifford
import qiskit.qasm2 as q2
from qiskit import QuantumCircuit
from IPython.display import display

from heisenbergUtilities import *

In [2]:
from qiskit_aer import AerSimulator
simulator = AerSimulator()

print(simulator.available_devices())

('CPU',)


Generate the linked list of all operations that take place in the inverse circuit

In [2]:
"""
LinkedListNode: A class that holds information related to a gate in a quantum circuit. 

PROPERTIES:
    - value (String): The name of the quantum gate
    - depth (int): The depth of the gate in the circuit
    - count (Int): The probability that the gate will meaningfully change the Pauli string.
    - next (LinkedListNode): The next gate in the circuit

"""

class LinkedListNode:
    def __init__(self, name="Initial", depth=0, count=0,  next=None):
        self.value = name
        self.depth = depth
        self.next = next
        self.count = count

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, name, depth):
        new_node = LinkedListNode(name, depth)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node

    def mark(self, depth):
        current_node = self.head
        while current_node:
            if current_node.depth == depth:
                current_node.count += 1
                return
            current_node = current_node.next

    def reset(self):
        current_node = self.head
        while current_node:
            current_node.count = 0
            current_node = current_node.next

<h2>Download MQT Benchmark circuits.</h2>

We'll start with DJ, RealAmpRandom, and GHZ since SB-QOPS caught those mutants 100% of the time for 5 qubit circuits



<h2>Generate mutants.</h2>

We'll start with mutants that add an unnecessary gate, then add mutants that replace a certain gate later

<h2> Define our CUT </h2>

This is where we will use SB-QOPS on a provided circuit and its mutants

For a single mutant, it took about 2 minutes to generate a 20 test suite (10 fail, 10 pass) on an RTX 4090 SUPER. 

There are 113 mutants for the DJ algorithm in use: 226 minutes or 3 3/4 hours

There are 89 mutants for the GHZ algorithm in use: 178 minutes or about 3 hours

There are 125 mutants for the RealAmpRandom algorithm in use: 250 minutes or just over 4 hours

BUT this cell should only need to be run once per algorithm with mutants under test. After it saves to the csv file at the end, this cell can be commented out as to not accidentally run it again. 

In the future if we want to test this methodology on higher qubit circuits or other algorithms, we'll likely either want to reduce the number of tests in the suite or the number of mutants we're analyzing

In [3]:
ORIGINAL_QASM = "dj_5_qubits.qasm"
ORIGINAL_QASM_FOLDER = "SBFL_circuits/MQTBench/"
QASM_MUTANT_FOLDER = "SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qubits/"
QUBITS = 5
Runs = 10
algo = "1+1"

correct_circuit = load_program(ORIGINAL_QASM, ORIGINAL_QASM_FOLDER).copy()
correct_list = LinkedList()
depth = 1

#Append each gate to the linked list
for instruction in correct_circuit.data:
    correct_list.append(instruction.name + " " + str(depth), depth)
    depth += 1


In [ ]:
import QOPS as qops
from QOPS_test import get_compact_program_specification_Z, get_mutants
from pathlib import Path

ga_result = pd.DataFrame(columns=['Program',"path_to_mutant",'catch_avg','avg_fitness','failing_testcases', 'passing_testcases'])
program_history = {}


# program_list = [x for x in os.listdir('benchmarkFilteration/benchmark2/') if x.split(".")[0].split("_")[-1] == QUBITS]
# for program_name in tqdm(program_list):

#program_specification = The compact program specification. SB-QOPS needs this to generate failing test cases
program_specification = get_compact_program_specification_Z(correct_circuit, shots=20000)

#mutants = A python list of qiskit QuantumCircuit variables with a mutation in them
mutants = []
mutants_names = []
root = Path(QASM_MUTANT_FOLDER)
for qasm_mutant in root.rglob("*.qasm"):
    mutants.append(load_program(qasm_mutant.name, QASM_MUTANT_FOLDER))
    mutants_names.append(qasm_mutant.name)

for mutant_index,mutant in enumerate(mutants):
    tester = qops.Circuit_Tester(CUT=mutant)
    tester.set_applicable_families_Z(program_specification)
    mutants_per_run = []
    fitness_per_run = []
    failing_testcases_per_run = []
    history_per_run = []

    print("NOW RUNNING TO FIND FAILING TESTS")
    #For a predefined number of attempts, try to find a set of failing test cases (output is above predefined tolerance)
    for runs in range(Runs):
        killed = 0
        pauli = {}
        fitness = 0
        for i in range(len(tester.applicable_families)):
            best_function,testcase, history = tester.run_mealoneplusone(i, 80)
            if best_function >0.1:
                killed = 1
                pauli = testcase
                fitness = best_function
                break
        mutants_per_run.append(killed)
        failing_testcases_per_run.append(pauli)
        fitness_per_run.append(fitness)
        history_per_run.append(history)

    avg_score = np.mean(mutants_per_run)
    avg_fitness = np.mean(fitness_per_run)

    #Here is our new, modified algorithm from the SB-QOPS method that attempts to find passing test cases as well. We'll need them for SBFL
    passing_testcases_per_run = []

    print("NOW RUNNING TO FIND PASSING TESTS")
    for runs in range(Runs):
        pauli = {}
        fitness = 0
        for i in range(len(tester.applicable_families)):
            best_function, testcase, history = tester.run_reverse_mealoneplusone(i,80)
            if best_function < 0.1:
                pauli = testcase
                break
        passing_testcases_per_run.append(pauli)

    #Replace static program file with "program_name" when ready to test fully
    """
    ga_result: A pandas dataframe with the following columns
        Program: Name of the mutant quantum circuit file
        Path_To_Mutant: Path to the mutant file
        Catch_Avg: The average number of times the mutant was identified by SB-QOPS
        Avg_Fitness: The average fitness of the search algorithm used. Higher usually indicates higher Catch_Avg
        Failing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a failing test case
        Passing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a passing test case
    """
    ga_result = pd.concat([pd.DataFrame([[mutants_names[mutant_index],QASM_MUTANT_FOLDER,avg_score,avg_fitness,json.dumps(failing_testcases_per_run), json.dumps(passing_testcases_per_run)]],columns=ga_result.columns),ga_result],ignore_index=True)
    ga_result.to_csv(f'dj_ga_result', index=False)

ga_result is a slightly altered pandas Dataframe with similar structure found in SB-QOPS's results folder. The only differences are changing the column "Program" to be the name of the mutant file instead of the original quantum circuit and changing "Mutant" to be the path to the mutant file instead of being an arbitrary index value

Now we want to run SBFL using Heisenberg evolution trees on each circuit placed in ga_result

Evolution analysis tends to take about 5 minutes for 10 failing tests, so about 10 minutes for 20 total tests

So about 1130 minutes for DJ, or 18.8 hours to evolve and analyze all 113 DJ mutants

About 890 minutes for GHZ, or 14.8 hours to evolve and analyze all 89 GHZ mutants

About 1250 minutes for RealAmpRandom, or 20.8 hours to evolve and analyze all 125 RealAmpRandom mutants

This is something to note in the final paper. Regardless of accuracy this methodology takes a long time to run. If results are promising, then we'll want to look into the tradeoffs between EXAM and hyperparameters to speed this up. Primarily the atol, max_terms, and search_step parameters in the evolve_pauli_exact method in HeisenbergUtilities.py


<h2>Heisenberg Evolutions</h2>

In [5]:
from heisenbergTree import *

ga_result = pd.read_csv("dj_ga_result")
tarantula_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
barinel_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])

#For each mutant of a circuit, run the SBFL implementation
for mutant, path in zip(ga_result.loc[:,"Program"], ga_result.loc[:,"path_to_mutant"]):
    print("Now evolving the following mutant: ", mutant)

    #Extract the raw test cases found for each mutant
    desired_failing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["failing_testcases"]]
    desired_passing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["passing_testcases"]]
    raw_failing_testcases = desired_failing_testcases["failing_testcases"].values[0].split("}")
    raw_passing_testcases = desired_passing_testcases["passing_testcases"].values[0].split("}")

    #Remove empty tests
    raw_failing_testcases = remove_null_tests(raw_failing_testcases)
    raw_passing_testcases = remove_null_tests(raw_passing_testcases)

    circuit_inverse = load_program(mutant,path).copy().inverse()  # Invert to track backward evolution of the operator
    forward_mutant = load_program(mutant, path).copy()

    #Create the linked list of operations in the inverse circuit
    operation_list = LinkedList()
    depth = 0

    #Append each gate to the linked list
    operation_list.append("Initial", depth)
    for instruction in circuit_inverse.data:
        depth += 1
        operation_list.append(instruction.name + " " + str((circuit_inverse.size()-depth)), depth)

    forward_list = LinkedList()
    depth = 1
    for instruction in forward_mutant.data:
        forward_list.append(instruction.name + " " + str(depth), depth)
        depth += 1

    #Create list of nodes in linked list. Somewhere down the road remove the linked list implementation. It's unnecessary and giving me a headache
    node_list = []
    node = operation_list.head
    while node:
        node_list.append(node)
        node = node.next

    #Perform Pauli propagation (Heisenberg evolution) for all test cases. Returns a dataframe with all counts for all operations
    global_frame_fail = heisenberg_evolve(circuit_inverse, operation_list, raw_failing_testcases, pass_fail = "fail")
    global_frame_pass = heisenberg_evolve(circuit_inverse, operation_list, raw_passing_testcases, pass_fail = "pass")

    global_frame = pd.concat([global_frame_fail, global_frame_pass], ignore_index=True)

    global_frame = global_frame.drop(["Initial"],axis=1)

    tarantula_scores = tarantula(global_frame)
    barinel_scores = barinel(global_frame)

    display(tarantula_scores)
    
    # Here is where analysis of the methodology begins. 
    # We first extract the position of the erroneous gate by comparing it to the original MQT gate
    # NOTE: This method is intended for single-added gates for now. Other types of mutants will require other methods later
    error_gate_location = find_erroneous_gate(forward_list, correct_list)

    print(error_gate_location)
    
    gates_observed = 0
    barinel_exam_score = 0
    for col_name, col_date in barinel_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            barinel_exam_score = gates_observed/(circuit_inverse.size()+1)
            break

    gates_observed = 0
    tarantula_exam_score = 0
    for col_name, col_date in tarantula_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            tarantula_exam_score = gates_observed/(circuit_inverse.size()+1)
            break

    barinel_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,barinel_exam_score]],columns=barinel_sbfl_result.columns),barinel_sbfl_result],ignore_index=True)
    barinel_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    barinel_sbfl_result.to_csv(f'dj_barinel_sbfl_result.csv', index=False)

    tarantula_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,tarantula_exam_score]],columns=tarantula_sbfl_result.columns),tarantula_sbfl_result],ignore_index=True)
    tarantula_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    tarantula_sbfl_result.to_csv(f'dj_tarantula_sbfl_result.csv', index=False)

barinel_sbfl_result
    

Now evolving the following mutant:  AddGate_p_inGap_7_.qasm


  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 10.37it/s]


,u 13,u 11,cx 10,p 9,h 6,cx 5,u 3,u 2,h 1,cx 12,h 8,cx 7,u 4,h 0
sum,0.614021,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587


h 1 h 1
h 2 h 2
u 3 u 3
u 4 u 4
u 5 u 5
cx 6 cx 6
h 7 h 7
cx 8 cx 8
h 9 h 9
cx 10 p 10
10
Now evolving the following mutant:  AddGate_sx_inGap_1_.qasm


...

h 1 h 1
h 2 h 2
u 3 u 3
u 4 u 4
u 5 u 5
cx 6 p 6
6
Now evolving the following mutant:  AddGate_ccx_inGap_4_.qasm


100%|██████████| 10/10 [00:01<00:00,  5.93it/s]


,u 13,cx 12,h 9,cx 8,h 7,ccx 6,cx 5,u 4,u 3,u 2,h 1,h 0,u 11,cx 10
sum,0.662976,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934


h 1 h 1
h 2 h 2
u 3 u 3
u 4 u 4
u 5 u 5
cx 6 cx 6
h 7 ccx 7
7


,Program,path_to_mutant,exam_score
0,AddGate_ccx_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.6
1,AddGate_p_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.533333
2,AddGate_s_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.333333
3,AddGate_h_inGap_6_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.133333
4,AddGate_h_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.666667
...,...,...,...
108,AddGate_x_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.866667
109,AddGate_cx_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.466667
110,AddGate_y_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.533333
111,AddGate_sx_inGap_1_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.8


In [7]:
tarantula_sbfl_result

,Program,path_to_mutant,exam_score
0,AddGate_ccx_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.785714
1,AddGate_p_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.857143
2,AddGate_s_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,1.0
3,AddGate_h_inGap_6_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.928571
4,AddGate_h_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,1.071429
...,...,...,...
108,AddGate_x_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.071429
109,AddGate_cx_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.857143
110,AddGate_y_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.857143
111,AddGate_sx_inGap_1_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.714286


<h2>Optimization Test and One-Shot Example</h2>

Using the pauli_prop package with marginally small atol and including max_terms/search_step yielded the same results in Ochiai as the old implementation, but ran twice as fast

In [ ]:
from pauli_prop import propagate_through_operator

MUTANT = "AddGate_p_inGap_7_.qasm"
MUTANT_PATH = "SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qubits/"

def evolve_pauli_exact(pauli_label, gate):
    """Return exact Pauli expansion after conjugation"""
    evolution = propagate_through_operator(pauli_label, gate, atol=1e-4, frame='h', max_terms = 80, search_step=3)

    return {
        p.to_label(): c for p, c in zip(evolution.paulis, evolution.coeffs)
    }

def load_program(name,path):
    qc = QuantumCircuit.from_qasm_file("{}/{}".format(path,name))
    qc.remove_final_measurements()
    if len(qc.clbits) > 0:
        for i in range(len(qc.clbits)):
            qc.measure(i, i)
    else:
        qc.measure_all()
    qc.remove_final_measurements()
    return qc.copy()

circuit_under_test = load_program(MUTANT, MUTANT_PATH)
circuit_inverse = circuit_under_test.copy().inverse()
output_from_csv = pd.read_csv("dj_ga_results.csv")
desired_testcases = output_from_csv.loc[(output_from_csv["Program"] == MUTANT), ["failing_testcases"]]
raw_testcases = desired_testcases["failing_testcases"].values[0].split("}")

for raw_idx, raw in enumerate(raw_testcases):
    if raw == ']':
        print("Caught one!", raw)
        raw_testcases.pop(raw_idx)

first = True

operation_list = LinkedList()
depth = 0
operation_list.append("Initial", depth)
for instruction in circuit_inverse.data:
    depth += 1
    operation_list.append(instruction.name + " " + str((circuit_inverse.size()-depth)), depth)


node_list = []
node = operation_list.head
while node:
    node_list.append(node)
    node = node.next

global_circuit_frame = pd.DataFrame({}, columns = [node.value for node in node_list] + ["pass/fail"], dtype=np.float64)

#For every provided test case, clean the string, convert it into a dictionary, and perform Pauli propagation
for test in tqdm(raw_testcases):

    #Cleaning up the test case string from the csv file to convert it into a dictionary
    test = test.replace("{", "")
    test = test.replace("]", "")
    test = test.replace("[", "")
    if not first:
        test = test[2:]
    test = "{" +test + "}"
    test = json.loads(test)

    #We want to keep track of the number of strings are in a specific test case and weight the results so that SBFL isn't biased by tests with more strings in them
    num_strings = 0
    
    #For each string specified in the test case, perform Pauli propagation through the circuit and store the results
    for pauli_string, pauli_coeff in zip(test.keys(), test.values()):

        #print("The coefficient related to string ", pauli_string, " is ", pauli_coeff)
        num_strings += 1

        # Define the initial Pauli operator to track
        # Qiskit layout is right-to-left: 'IX' means X acts on Qubit 0, I acts on Qubit 1
        initial_pauli = SparsePauliOp(pauli_string)  
        prev_pauli = initial_pauli
        num_qubits = circuit_inverse.num_qubits

        transition_graph = {}  # step -> list of (from, to, coeff)

        #Define the current layer of the evolution tree, starting with the initial Pauli having a probability of 100% to appear
        current_layer = {
            initial_pauli.paulis[0].to_label(): 1.0  # Start with the initial Pauli having probability 1
        }

        #For each gate in the quantum circuit, propagate backwards. The circuit is already inversed 
        for step_idx, (instruction, qargs, cargs) in enumerate(circuit_inverse.data, start=1):

            #Define operation being performed
            gate_type = instruction.name 
            gate_op = Operator(instruction)

            #Create a do-nothing operator, then add the desired gate to the do-nothing to only act on the desired qubits
            q_indices = [qarg._index for qarg in qargs]
            full_gate_op = Operator.from_label("I" * num_qubits).compose(gate_op, q_indices)

            new_layer = {}
            edges = []

            #For each Pauli in the current layer of the evolution tree, apply the desired operator to the desired qubits
            for pauli_in, prob_in in current_layer.items():

                evolved_dict = evolve_pauli_exact(SparsePauliOp(pauli_in), SparsePauliOp.from_operator(full_gate_op))

                #For each resulting Pauli string, calculate the probability of the inverse circuit reaching this state
                #This will be used later to weight the counts for SBFL
                for pauli_out, coeff_out in evolved_dict.items():

                    total_prob = prob_in * ((coeff_out.real)**2)

                    # accumulate node weights
                    new_layer[pauli_out] = (new_layer.get(pauli_out, 0) + total_prob)

                    # store exact edge
                    edges.append({
                        "from": pauli_in,
                        "to": pauli_out,
                        "weight": coeff_out,
                        "probability": total_prob
                    })

            #Simplify numerical noise, we don't care about states that have such a low chance of occuring that it's negligible
            current_layer = {
                pauli: prob for pauli, prob in new_layer.items() if abs(prob) > 1e-4
            }

            transition_graph[step_idx] = {
                "gate_type": gate_type,
                "edges": edges
            }

        #Start at first gate in the inverse circuit
        checked_gate = operation_list.head.next
        idx = 1

        #For each gate in the circuit
        while checked_gate:

            #For each transition through that gate
            for edge in transition_graph[idx]["edges"]:
                
                #If the previous Pauli string evolved into a new Pauli string, 
                #add the probability of that new string occuring to that gate's count
                if edge["from"] != edge["to"]:
                    checked_gate.count += edge["probability"]
            
            #We weight the count by the coefficient related to the Pauli string that SB-QOPS provides
            checked_gate.count *= abs(pauli_coeff)
            idx += 1
            checked_gate = checked_gate.next

    first = False
    node = operation_list.head
    gate_list = []
    while node:
        gate_list.append(node.count/num_strings)
        node = node.next
    gate_list.append("fail")
    global_circuit_frame = pd.concat([global_circuit_frame, pd.DataFrame([gate_list], columns=global_circuit_frame.columns)], ignore_index=True)
    operation_list.reset()

global_circuit_frame
    


In [ ]:
global_circuit_frame = global_circuit_frame.drop(["Initial"],axis=1)
global_circuit_frame

<h2>Tarantula Algorithm</h2>

NOTE: This algorithm will not run correctly until SB-QOPS is configured to return a selection of passing tests on top of the failing tests it already provides

In [ ]:
num_fail_tests = len(global_circuit_frame[global_circuit_frame["pass/fail"] == "fail"])
num_pass_tests = len(global_circuit_frame[global_circuit_frame["pass/fail"] == "pass"])
fail_counts = global_circuit_frame[global_circuit_frame["pass/fail"] == "fail"].agg(["sum"]).drop(["pass/fail"], axis=1)
pass_counts = global_circuit_frame[global_circuit_frame["pass/fail"] == "pass"].agg(["sum"]).drop(["pass/fail"], axis=1)

tarantula_scores = (fail_counts/num_fail_tests)/((fail_counts/num_fail_tests)+(pass_counts/num_pass_tests))
tarantula_scores

<h2>Ochiai Algorithm</h2>

NOTE: This algorithms performance is also degraded until SB-QOPS also returns a selection of passing tests

In [ ]:
pass_counts = global_circuit_frame[global_circuit_frame["pass/fail"] == "pass"].agg(["sum"]).drop(["pass/fail"], axis=1)
fail_counts = global_circuit_frame[global_circuit_frame["pass/fail"] == "fail"].agg(["sum"]).drop(["pass/fail"], axis=1)
num_fail_tests = len(global_circuit_frame[global_circuit_frame["pass/fail"] == "fail"])

ochiai_scores = fail_counts/np.sqrt(num_fail_tests*(fail_counts+pass_counts))
ochiai_scores = ochiai_scores[ochiai_scores.iloc[0].sort_values(ascending=False).index]
ochiai_scores
